# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

In [ ]:
# List all record sets with their `@id`, name, and available fields/columns

print("Available Record Sets:")
for rs in metadata.record_sets:
    print(f"- @id: {rs.id} | name: {rs.name}")
    
    print("  Fields:")
    for field in rs.fields:
        print(f"    * @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    if hasattr(rs, 'columns'):
        print("  Columns:")
        for col in rs.columns:
            print(f"    * @id: {col.id} | name: {col.name} | dataType: {col.data_type}")
    print("\n")

if len(metadata.record_sets) == 0:
    print("No record sets defined directly in top-level Croissant metadata. Attempting to infer from dataset contents.")
    # To explore, try to fetch one record set from the known file distribution, if possible:
    print("Calling dataset.records() without specifying record_set to see what is available:")
    try:
        sample_records = list(dataset.records())
        if len(sample_records) > 0:
            print("Sample record keys:", list(sample_records[0].keys()))
    except Exception as e:
        print("Error retrieving records:", e)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Since the Croissant metadata did not enumerate explicit record sets with @id, we can attempt to extract all records
# by calling dataset.records() with no arguments. Alternatively, if you identified an @id in the previous cell, use it below.

all_records = list(dataset.records())
df = pd.DataFrame(all_records)
print(f"Loaded {len(df)} records. Columns available:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Note:** All references to fields and columns in this section use their `@id` values as column names, ensuring clear correspondence to the Croissant schema.

In [ ]:
# Example EDA using numeric and group fields present in the dataset
# Find possible numeric fields by data type or by inspecting column names
numeric_candidates = [c for c in df.columns if any(x in c.lower() for x in ['abundance', 'mw', 'count', 'coverage', 'weight', 'intensity', 'area'])]
print('Possible numeric fields:', numeric_candidates)

# Select a field by @id known to be numeric, fallback to the first one
numeric_field = numeric_candidates[0] if numeric_candidates else df.columns[0]

# Choose a threshold for filtering
threshold = df[numeric_field].quantile(0.75) if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    filtered_df = df[df[numeric_field] > threshold]
else:
    print(f"Field {numeric_field} is not numeric. Unable to filter by threshold.")
    filtered_df = df
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
else:
    print(f"Cannot normalize non-numeric field {numeric_field}.")

# Try grouping by a suitable categorical field
group_candidates = [c for c in df.columns if c != numeric_field and df[c].nunique() < len(df) / 2 and df[c].dtype == 'object']
group_field = group_candidates[0] if group_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean values by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

# If grouping field found, boxplot by group
if group_field:
    plt.figure(figsize=(12, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
In this notebook, we explored the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the `mlcroissant` library. We loaded the metadata and records, reviewed available fields and columns by their `@id`, and performed basic exploratory data analysis including filtering, normalization, grouping, and visualization of a numeric field.

This dataset provides detailed mass spectrometry protein measurements. For downstream applications such as biomarker discovery or comparative studies, advanced analysis such as statistical inference and domain-specific feature engineering is recommended.
